# Chapter 2: LLM API Basics

Companion notebook for Chapter 2. Each section corresponds to a listing in the book.

Run the code cells from top to bottom in a fresh kernel after completing the
README setup and this chapter's API prerequisites.


## Setup (Listing 2.1)

In [1]:
from pathlib import Path
from dotenv import load_dotenv

# Locate this checkout, including when Jupyter starts in a chapter folder.
PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "scratch_agents").is_dir()
)
import sys
# Use this checkout even if the selected kernel was installed elsewhere.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

%load_ext autoreload
%autoreload 2


In [11]:
%reload_ext autoreload

## Gemini API

In [ ]:
from litellm import completion

response = completion(
    model="gemini/gemini-2.5-flash",
    messages=[
        {"role": "system", "content": "你是一位 Python 老师，请用中文回答。"},
        {"role": "user", "content": "用一个例子解释 Python 的生成器。"},
    ],
)

print(response.choices[0].message.content)
print(response.usage)  # token 用量

## OpenAI API (Listing 2.2)

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(response.choices[0].message.content)

## Anthropic API (Listing 2.3)

In [ ]:
from anthropic import Anthropic

client = Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(response.content[0].text)

## Unified API with LiteLLM (Listing 2.4)

In [ ]:
from litellm import completion

response = completion(
    model="gemini/gemini-2.5-flash",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)

response = completion(
    model="claude-sonnet-4-6",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)

## Demonstrating API statelessness (Listing 2.5)

In [ ]:
from litellm import completion

# First call
response1 = completion(
    model="gemini/gemini-2.5-flash",
    messages=[{"role": "user", "content": "My name is Jungjun."}]
)
print(response1.choices[0].message.content)

# Second call
response2 = completion(
    model="gemini/gemini-2.5-flash",
    messages=[{"role": "user", "content": "What is my name?"}]
)
print(response2.choices[0].message.content)

## Managing conversation history (Listing 2.6)

In [ ]:
from litellm import completion

messages = []

# First exchange
messages.append({"role": "user", "content": "My name is Jungjun."})
response1 = completion(model="gemini/gemini-2.5-flash", messages=messages)
assistant_message1 = response1.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_message1})
print(assistant_message1)

# Second exchange - includes previous conversation history
messages.append({"role": "user", "content": "What is my name?"})
response2 = completion(model="gemini/gemini-2.5-flash", messages=messages)
assistant_message2 = response2.choices[0].message.content
print(assistant_message2)

## Structured output with Pydantic (Listing 2.7)

In [ ]:
from pydantic import BaseModel
from litellm import completion

class ExtractedInfo(BaseModel):
    name: str
    email: str
    phone: str | None = None

response = completion(
    model="gemini/gemini-2.5-flash",
    messages=[{
        "role": "user",
        "content": "My name is John Smith, my email is john@example.com, and my phone is 555-1234."
    }],
    response_format=ExtractedInfo
)

result = response.choices[0].message.content
print(result)

## Asynchronous LLM calls (Listing 2.8)

In [ ]:
import asyncio
from litellm import acompletion

async def get_response(prompt: str) -> str:
    response = await acompletion(
        model="gemini/gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

prompts = [
    "What is 2 + 2?",
    "What is the capital of Japan?",
    "Who wrote Romeo and Juliet?"
]

# Execute all requests concurrently
tasks = [get_response(p) for p in prompts]
results = await asyncio.gather(*tasks)

for prompt, result in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {result}\n")

## Rate-limited concurrent requests (Listing 2.9)

In [ ]:
import asyncio
from litellm import acompletion

# Limit to 10 concurrent requests
semaphore = asyncio.Semaphore(10)

async def call_llm(prompt: str) -> str:
    """LLM call with rate limiting and automatic retry."""
    async with semaphore:
        response = await acompletion(
            model="gemini/gemini-2.5-flash",
            messages=[{"role": "user", "content": prompt}],
            num_retries=3  # Automatic retry with exponential backoff
        )
        return response.choices[0].message.content

# Even with 100 concurrent tasks, only 10 API calls run at a time
prompts = [f"What is {i} + {i}?" for i in range(100)]
tasks = [call_llm(p) for p in prompts]
results = await asyncio.gather(*tasks, return_exceptions=True)

successes = sum(1 for r in results if not isinstance(r, Exception))
print(f"Successful: {successes}/{len(results)}")

## GAIA Benchmark Evaluation

Listings 2.10–2.17 are implemented in `scratch_agents.eval.gaia`. The notebook
imports the orchestration function and runs the experiment. Read the source
of `scratch_agents/eval/gaia.py` alongside the book listings for the full
implementation.

### Loading GAIA Level 1 problems (Listing 2.10)

In [2]:
# GAIA is gated: accept access on Hugging Face and set HF_TOKEN first.
from datasets import load_dataset

level1_problems = load_dataset(
    "gaia-benchmark/GAIA",
    "2023_level1",
    split="validation")

print(f"Number of Level 1 problems: {len(level1_problems)}")
for index, problem in enumerate(level1_problems, start=1):
    print(f"\nProblem {index}:")
    for field, value in problem.items():
        print(f"{field}: {value}")


Number of Level 1 problems: 53

Problem 1:
task_id: e1fc63a2-da7a-432f-be78-7c4a95598703
Question: If Eliud Kipchoge could maintain his record-making marathon pace indefinitely, how many thousand hours would it take him to run the distance between the Earth and the Moon its closest approach? Please use the minimum perigee value on the Wikipedia page for the Moon when carrying out your calculation. Round your result to the nearest 1000 hours and do not use any comma separators if necessary.
Level: 1
Final answer: 17
file_name: 
file_path: 
Annotator Metadata: {'Steps': '1. Googled Eliud Kipchoge marathon pace to find 4min 37sec/mile\n2. Converted into fractions of hours.\n3. Found moon periapsis in miles (225,623 miles).\n4. Multiplied the two to find the number of hours and rounded to the nearest 100 hours.', 'Number of steps': '4', 'How long did this take?': '20 Minutes', 'Tools': '1. A web browser.\n2. A search engine.\n3. A calculator.', 'Number of tools': '3'}

Problem 2:
task_id: 

### Model selection and execution (Listing 2.17)

In [7]:
from scratch_agents.eval.gaia import run_experiment

MODELS = [
    # "gpt-5.5",
    "gemini/gemini-2.5-flash",
    # "anthropic/claude-sonnet-4-6",
    # "anthropic/claude-haiku-4-5"
]

subset = level1_problems.select(range(20))
results = await run_experiment(subset, MODELS)

# Look up question text, which is not included in evaluation results.
questions_by_id = {
    problem["task_id"]: (number, problem["Question"])
    for number, problem in enumerate(subset, start=1)
}

# Per-model accuracy and questions judged solvable by each model
for model, model_results in results.items():
    correct = sum(1 for r in model_results if r["correct"])
    solvable = sum(1 for r in model_results if r.get("is_solvable"))
    total = len(model_results)
    print(f"{model}: {correct}/{total} correct, {solvable}/{total} judged solvable")
    for r in model_results:
        if r.get("is_solvable"):
            number, question = questions_by_id[r["task_id"]]
            print(f"\nQuestion #{number} (task_id: {r['task_id']})")
            print(f"Question: {question}")
            print(f"Model answer: {r['prediction']}")
            print(f"Expected answer: {r['answer']}")


  0%|          | 0/20 [00:00<?, ?it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.I

 65%|██████▌   | 13/20 [00:01<00:00,  9.38it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.I

100%|██████████| 20/20 [00:03<00:00,  5.06it/s]

gemini/gemini-2.5-flash: 1/20 correct, 1/20 judged solvable

Question #19 (task_id: 4fc2f1ae-8625-45b5-ab34-ad4433bc21f8)
Question: Who nominated the only Featured Article on English Wikipedia about a dinosaur that was promoted in November 2016?
Model answer: FunkMonk
Expected answer: FunkMonk
